# Comparaison 6 modèles — CWRU Bearing Dataset — by_severity

| Champ | Valeur |
|-------|--------|
| **Scénario** | by_severity : 0.007" → 0.014" → 0.021" (3 tâches — drift de sévérité progressif) |
| **Modèles** | EWC · HDC · TinyOL · KMeans · Mahalanobis · DBSCAN |
| **Dataset** | CWRU Bearing Dataset (CWRU) — 9 features statistiques |
| **Sprint** | 12 — S12-08 |

Ce notebook agrège les résultats des expériences **exp_080, exp_081, exp_082, exp_083, exp_084, exp_085**.

**Figures générées** :
1. `comparison_aa_af_bwt.png` — Barplot groupé AA/AF/BWT (6 modèles)
2. `scatter_af_vs_ram.png` — Scatter AF vs RAM (trade-off oubli/contrainte embarquée)
3. `cross_scenario_af_comparison.png` — AF(by_severity) vs AF(by_fault_type) par modèle
4. `ranking_models.png` — Ranking par score composite (AA − AF)

In [ ]:
# Section 1 — Setup + chargement normalisé des 6 modèles (exp_080–085)
import json
import os
import sys
from datetime import datetime
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, Markdown, display

# --- CWD navigation ---
_cwd = Path(".").resolve()
if _cwd.name == "cwru_by_severity":
    os.chdir(_cwd.parent.parent.parent)
elif _cwd.name == "cl_eval":
    os.chdir(_cwd.parent.parent)
elif _cwd.name == "notebooks":
    os.chdir(_cwd.parent)
REPO_ROOT = Path(".").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.evaluation.plots import plot_metrics_comparison, save_figure

FIGURES_DIR = REPO_ROOT / "notebooks/figures/cl_evaluation/comparison/cwru/by_severity"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

TASK_NAMES  = ['0.007"', '0.014"', '0.021"']
MODEL_ORDER = ["EWC", "HDC", "TinyOL", "KMeans", "Mahalanobis", "DBSCAN"]
RAM_BUDGET  = 65_536

MODEL_EXP_MAP = {
    "EWC":         "exp_080_ewc_cwru_by_severity",
    "HDC":         "exp_081_hdc_cwru_by_severity",
    "TinyOL":      "exp_082_tinyol_cwru_by_severity",
    "KMeans":      "exp_083_kmeans_cwru_by_severity",
    "Mahalanobis": "exp_084_mahalanobis_cwru_by_severity",
    "DBSCAN":      "exp_085_dbscan_cwru_by_severity",
}

FAULT_TYPE_EXP_MAP = {
    "EWC":         "exp_074_ewc_cwru_by_fault_type",
    "HDC":         "exp_075_hdc_cwru_by_fault_type",
    "TinyOL":      "exp_076_tinyol_cwru_by_fault_type",
    "KMeans":      "exp_077_kmeans_cwru_by_fault_type",
    "Mahalanobis": "exp_078_mahalanobis_cwru_by_fault_type",
    "DBSCAN":      "exp_079_dbscan_cwru_by_fault_type",
}

BASELINE_EXP_MAP = {
    "EWC":         "exp_068_ewc_cwru_single_task",
    "HDC":         "exp_069_hdc_cwru_single_task",
    "TinyOL":      "exp_070_tinyol_cwru_single_task",
    "KMeans":      "exp_071_kmeans_cwru_single_task",
    "Mahalanobis": "exp_072_mahalanobis_cwru_single_task",
    "DBSCAN":      "exp_073_dbscan_cwru_single_task",
}


def load_acc_matrix(raw_mat: list) -> np.ndarray:
    mat = np.full((3, 3), np.nan, dtype=float)
    for i, row in enumerate(raw_mat):
        for j, v in enumerate(row):
            if v is not None:
                mat[i, j] = v
    return mat


BASE = REPO_ROOT / "experiments"
results      = {}
acc_matrices = {}
baselines    = {}

for model in MODEL_ORDER:
    exp_dir      = BASE / MODEL_EXP_MAP[model] / "results"
    metrics_path = exp_dir / "metrics_cl.json"
    if not metrics_path.exists():
        print(f"⚠️  {MODEL_EXP_MAP[model]} non disponible — mock activé")
        results[model] = {"aa": 0.0, "af": 0.0, "bwt": 0.0,
                          "ram_peak_bytes": 0, "inference_latency_ms": 0.0,
                          "n_params": 0, "per_task_acc": [0.0, 0.0, 0.0]}
        acc_matrices[model] = np.zeros((3, 3))
        continue
    raw = json.loads(metrics_path.read_text())
    results[model] = {
        "aa":  raw["acc_final"],
        "af":  raw["avg_forgetting"],
        "bwt": raw["backward_transfer"],
        "ram_peak_bytes":       raw["ram_peak_bytes"],
        "inference_latency_ms": raw["inference_latency_ms"],
        "n_params":    raw["n_params"],
        "per_task_acc": raw["per_task_acc"],
    }
    acc_matrices[model] = load_acc_matrix(raw["acc_matrix"])

    bline_path = BASE / BASELINE_EXP_MAP[model] / "results" / "metrics_single_task.json"
    if bline_path.exists():
        braw = json.loads(bline_path.read_text())
        baselines[model] = braw.get("accuracy") or braw.get("acc_final", float("nan"))
    else:
        baselines[model] = float("nan")

    r = results[model]
    print(f"{model:12s} → AA={r['aa']:.4f} AF={r['af']:.4f} BWT={r['bwt']:+.4f} "
          f"RAM={r['ram_peak_bytes']/1024:5.1f}Ko lat={r['inference_latency_ms']:.5f}ms n={r['n_params']}")

print(f"\n6 modèles chargés | {datetime.now():%Y-%m-%d %H:%M}")

In [ ]:
# Section 2 — Tableau comparatif AA / AF / BWT / RAM / latence (6 modèles)

RAM_LIMIT = 65_536
header = "| Modèle | AA ↑ | AF ↓ | BWT | RAM | Latence | n_params | Baseline ST |"
sep    = "|--------|:----:|:----:|:---:|:---:|:-------:|:--------:|:-----------:|"
rows   = [header, sep]

for model in MODEL_ORDER:
    r     = results[model]
    ram_b = r["ram_peak_bytes"]
    ram_s = f"{ram_b/1024:.1f} Ko{'  ⚠️' if ram_b > RAM_LIMIT else ''}"
    bst   = baselines.get(model, float("nan"))
    bst_s = f"{bst:.4f}" if not (isinstance(bst, float) and bst != bst) else "—"
    line  = (
        f"| {model} | {r['aa']:.4f} | {r['af']:.4f} | {r['bwt']:+.4f} | "
        f"{ram_s} | {r['inference_latency_ms']:.5f} ms | {r['n_params']} | {bst_s} |"
    )
    rows.append(line)

display(Markdown("### Tableau comparatif — 6 modèles CL (cwru/by_severity)\n\n" + "\n".join(rows)))

In [ ]:
# Section 3 — Barplot groupé AA / AF / BWT (6 modèles) → comparison_aa_af_bwt.png

fig = plot_metrics_comparison(
    results,
    metrics=["aa", "af", "bwt"],
    title="AA / AF / BWT — CWRU/by_severity (6 modèles)",
)
save_figure(fig, FIGURES_DIR / "comparison_aa_af_bwt.png")
display(Image(str(FIGURES_DIR / "comparison_aa_af_bwt.png")))

In [ ]:
# Section 4 — Scatter AF vs RAM → scatter_af_vs_ram.png

SCATTER_MARKERS = {
    "EWC":         ("o", "#1f77b4"),
    "HDC":         ("s", "#ff7f0e"),
    "TinyOL":      ("^", "#2ca02c"),
    "KMeans":      ("D", "#d62728"),
    "Mahalanobis": ("P", "#9467bd"),
    "DBSCAN":      ("*", "#8c564b"),
}

fig, ax = plt.subplots(figsize=(8, 5))

ax.axvspan(0, RAM_BUDGET / 1024, alpha=0.07, color="green", label=f"Zone STM32 ≤ 64 Ko")
ax.axvline(RAM_BUDGET / 1024, color="red", linestyle="--", linewidth=1.5, label="Budget 64 Ko")

for name in MODEL_ORDER:
    r = results[name]
    ram_kb = r["ram_peak_bytes"] / 1024
    af_val = r["af"]
    marker, color = SCATTER_MARKERS[name]
    ax.scatter(ram_kb, af_val, marker=marker, color=color, s=130, zorder=5, label=name,
               edgecolor="black", linewidth=0.5)
    ax.annotate(name, xy=(ram_kb, af_val), xytext=(ram_kb * 1.04, af_val + 0.001), fontsize=9)

ax.set_xlabel("RAM peak (Ko)", fontsize=11)
ax.set_ylabel("AF (Average Forgetting) ↓", fontsize=11)
ax.set_title(
    "Trade-off embarqué : RAM vs. Oubli\n(CWRU/by_severity — Gap 2 STM32 ≤ 64 Ko)",
    fontsize=12, fontweight="bold",
)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
fig.tight_layout()
save_figure(fig, FIGURES_DIR / "scatter_af_vs_ram.png")
display(Image(str(FIGURES_DIR / "scatter_af_vs_ram.png")))

In [ ]:
# Section 5 — Cross-scenario : AF(by_severity) vs AF(by_fault_type) par modèle
# Hypothèse : drift de sévérité (doux) → oubli moindre qu'un changement de type de défaut
# → cross_scenario_af_comparison.png

BASE2 = BASE  # même REPO_ROOT / "experiments"
af_severity   = []
af_fault_type = []
models_available = []

for model in MODEL_ORDER:
    af_sev = results[model]["af"]
    ft_path = BASE2 / FAULT_TYPE_EXP_MAP[model] / "results" / "metrics_cl.json"
    if ft_path.exists():
        ft_raw = json.loads(ft_path.read_text())
        af_ft  = ft_raw["avg_forgetting"]
    else:
        af_ft = float("nan")
    af_severity.append(af_sev)
    af_fault_type.append(af_ft)
    models_available.append(model)

x = np.arange(len(models_available))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
bars1 = ax.bar(x - width / 2, af_severity,   width, label="by_severity (drift doux)",
               color="#FB8C00", edgecolor="black", linewidth=0.6)
bars2 = ax.bar(x + width / 2, af_fault_type, width, label="by_fault_type (drift dur)",
               color="#1565C0", edgecolor="black", linewidth=0.6)

for bar, val in zip(bars1, af_severity):
    if val == val:  # not nan
        ax.text(bar.get_x() + bar.get_width() / 2, val + 0.002, f"{val:.3f}",
                ha="center", va="bottom", fontsize=8)
for bar, val in zip(bars2, af_fault_type):
    if val == val:
        ax.text(bar.get_x() + bar.get_width() / 2, val + 0.002, f"{val:.3f}",
                ha="center", va="bottom", fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(models_available, fontsize=10)
ax.set_ylabel("AF (Average Forgetting) ↓", fontsize=11)
ax.set_title(
    "Cross-scenario : AF(by_severity) vs AF(by_fault_type)\n"
    "Un AF plus faible en by_severity valide l'hypothèse de drift doux",
    fontsize=12, fontweight="bold",
)
ax.axhline(y=0, color="gray", linestyle="--", linewidth=0.8, alpha=0.5)
ax.legend(fontsize=10)
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
save_figure(fig, FIGURES_DIR / "cross_scenario_af_comparison.png")
display(Image(str(FIGURES_DIR / "cross_scenario_af_comparison.png")))

# Résumé textuel de l'hypothèse
print("\nHypothèse : AF(severity) < AF(fault_type) par modèle")
print("-" * 52)
for model, af_s, af_f in zip(models_available, af_severity, af_fault_type):
    if af_s == af_s and af_f == af_f:
        verdict = "✅ VALIDÉE" if af_s < af_f else "❌ NON VALIDÉE"
        print(f"  {model:12s}: AF_sev={af_s:.4f}  AF_ft={af_f:.4f}  {verdict}")
    else:
        print(f"  {model:12s}: données manquantes")

In [ ]:
# Section 6 — Ranking des modèles par score composite (AA − AF) → ranking_models.png

scores = {model: results[model]["aa"] - results[model]["af"] for model in MODEL_ORDER}
sorted_models = sorted(scores, key=scores.get, reverse=True)
sorted_scores = [scores[m] for m in sorted_models]

RANK_COLORS = {
    "EWC":         "#1f77b4",
    "HDC":         "#ff7f0e",
    "TinyOL":      "#2ca02c",
    "KMeans":      "#d62728",
    "Mahalanobis": "#9467bd",
    "DBSCAN":      "#8c564b",
}

fig, ax = plt.subplots(figsize=(8, 4))
colors_bar = [RANK_COLORS[m] for m in sorted_models]
bars = ax.barh(sorted_models, sorted_scores, color=colors_bar, edgecolor="black", linewidth=0.5)

for bar, score in zip(bars, sorted_scores):
    ax.text(
        score + 0.002, bar.get_y() + bar.get_height() / 2,
        f"{score:.4f}", va="center", ha="left", fontsize=9,
    )

ax.set_xlabel("Score composite (AA − AF)", fontsize=11)
ax.set_title(
    "Ranking des modèles CL — cwru/by_severity\n(Score = AA − AF : performance nette de l'oubli)",
    fontsize=12, fontweight="bold",
)
ax.set_xlim(0, max(sorted_scores) * 1.15)
ax.grid(axis="x", alpha=0.3)
ax.invert_yaxis()
fig.tight_layout()
save_figure(fig, FIGURES_DIR / "ranking_models.png")
display(Image(str(FIGURES_DIR / "ranking_models.png")))

print("\nCritères d'acceptation (S12-08) :")
for fig_name in ["comparison_aa_af_bwt.png", "scatter_af_vs_ram.png",
                 "cross_scenario_af_comparison.png", "ranking_models.png"]:
    p = FIGURES_DIR / fig_name
    print(f"  [{'OK' if p.exists() else 'MANQUANTE'}] {fig_name}")